# Notebook 2 — Classical ML: Baseline vs Learned Embeddings

**Goal**: Compare Linear Regression, Random Forest and XGBoost under two feature regimes:

| Regime | Categorical encoding |
|--------|----------------------|
| **Baseline** | OHE for `original_language` + multi-hot for `genres` (sparse, no relational info) |
| **Embeddings** | Dense vectors loaded from NB1 — genre mean-pool + lang lookup (rich, learned) |

**Final table**: 7 models — 6 classical (3 algos × 2 regimes) + EmbeddingANN from NB1.


## 1. Setup & Imports

In [ ]:
import os, json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model  import LinearRegression
from sklearn.ensemble      import RandomForestRegressor
from xgboost               import XGBRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute        import SimpleImputer
from sklearn.metrics       import mean_squared_error, mean_absolute_error
from scipy.sparse          import csr_matrix, hstack
import warnings; warnings.filterwarnings('ignore')


## 2. Paths

In [ ]:
DATA_DIR = r"C:\Users\Shivanshu Sirohi\OneDrive\Desktop\Shivanshu\White Paper\Personalization\MovieLens\data"
EMB_DIR  = r"C:\Users\Shivanshu Sirohi\OneDrive\Desktop\Shivanshu\White Paper\Personalization\MovieLens\embeddings"

# Confirm embedding files from NB1 exist before proceeding
for fname in ['genre_embeddings.npy', 'lang_embeddings.npy',
              'genre_encoder.json',    'lang_encoder.json', 'ann_results.json']:
    fpath = os.path.join(EMB_DIR, fname)
    status = 'OK' if os.path.exists(fpath) else '*** MISSING — run NB1 first ***'
    print(f"  {status} | {fname}")


## 3. Load Data

In [ ]:
df_train = pd.read_csv(DATA_DIR + '\\train.csv')
df_val   = pd.read_csv(DATA_DIR + '\\val.csv')
df_test  = pd.read_csv(DATA_DIR + '\\test.csv')

TARGET   = 'rating'
CAT_GENRE= 'genres'
CAT_LANG = 'original_language'
NUM_COLS = [
    'imdbRating', 'imdbVotes',
    'tmdbRating', 'tmdbVotes',
    'popularity', 'budget', 'runtime', 'revenue',
    'tag_count',  'avg_relevance', 'max_relevance'
]

print(f"Train {df_train.shape} | Val {df_val.shape} | Test {df_test.shape}")


## 4. Numerical Preprocessing

**Identical to NB1** so both experiments sit on the same numerical foundation.
Tree models don't need scaling, but we scale anyway to keep the comparison clean
(the numerical columns are the same regardless — scaling only changes magnitude,
not the tree split logic).


In [ ]:
# Tag features: missing = no tag activity → 0
for df_ in [df_train, df_val, df_test]:
    df_[['tag_count', 'avg_relevance', 'max_relevance']] =         df_[['tag_count', 'avg_relevance', 'max_relevance']].fillna(0)

# Median imputation (fit on train only)
imputer = SimpleImputer(strategy='median')
df_train[NUM_COLS] = imputer.fit_transform(df_train[NUM_COLS])
df_val[NUM_COLS]   = imputer.transform(df_val[NUM_COLS])
df_test[NUM_COLS]  = imputer.transform(df_test[NUM_COLS])

# log1p
for df_ in [df_train, df_val, df_test]:
    df_[NUM_COLS] = np.log1p(df_[NUM_COLS].clip(lower=0))

# StandardScaler (fit on train only)
scaler = StandardScaler()
df_train[NUM_COLS] = scaler.fit_transform(df_train[NUM_COLS])
df_val[NUM_COLS]   = scaler.transform(df_val[NUM_COLS])
df_test[NUM_COLS]  = scaler.transform(df_test[NUM_COLS])

# Labels
y_train = df_train[TARGET].values
y_val   = df_val[TARGET].values
y_test  = df_test[TARGET].values

print("Numerical preprocessing done.")


## 5. Regime A — Baseline Feature Matrix

Categorical encoding: **OHE** for language + **multi-hot** for genres.
This is the standard approach — sparse, no learned structure.


In [ ]:
def build_baseline_matrix(df_tr, df_v, df_te):
    """
    Build sparse feature matrices using OHE (language) + multi-hot (genres).
    Fit encoders on train only.
    Returns X_train, X_val, X_test (scipy sparse), plus the fitted encoders.
    """
    # Multi-hot genres
    genres_tr = df_tr[CAT_GENRE].fillna('Unknown').str.get_dummies('|')
    genre_cols = genres_tr.columns
    genres_v  = df_v[CAT_GENRE].fillna('Unknown').str.get_dummies('|').reindex(columns=genre_cols, fill_value=0)
    genres_te = df_te[CAT_GENRE].fillna('Unknown').str.get_dummies('|').reindex(columns=genre_cols, fill_value=0)

    # OHE language
    ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
    lang_tr = ohe.fit_transform(df_tr[[CAT_LANG]].fillna('unknown'))
    lang_v  = ohe.transform(df_v[[CAT_LANG]].fillna('unknown'))
    lang_te = ohe.transform(df_te[[CAT_LANG]].fillna('unknown'))

    # Numeric (already scaled)
    num_tr = csr_matrix(df_tr[NUM_COLS].values)
    num_v  = csr_matrix(df_v[NUM_COLS].values)
    num_te = csr_matrix(df_te[NUM_COLS].values)

    X_tr = hstack([num_tr, lang_tr, csr_matrix(genres_tr.values)])
    X_v  = hstack([num_v,  lang_v,  csr_matrix(genres_v.values)])
    X_te = hstack([num_te, lang_te, csr_matrix(genres_te.values)])

    feat_names = (NUM_COLS
                  + ohe.get_feature_names_out([CAT_LANG]).tolist()
                  + list(genre_cols))
    return X_tr, X_v, X_te, feat_names

X_train_base, X_val_base, X_test_base, feat_names_base = build_baseline_matrix(
    df_train, df_val, df_test
)
print(f"Baseline feature matrix — train: {X_train_base.shape}")
print(f"Total features: {X_train_base.shape[1]}  "
      f"({len(NUM_COLS)} numeric + {X_train_base.shape[1]-len(NUM_COLS)} categorical)")


## 6. Regime B — Embedding Feature Matrix

Load the saved weights and encoders from NB1.
For each row:
- **Language**: look up `lang_encoder` → integer index → row in `lang_embeddings.npy`
- **Genres**: split by `|` → look up each token → stack embedding rows → mean

These dense vectors replace the sparse OHE/multi-hot columns entirely.


In [ ]:
# ── Load saved artifacts ──
genre_weights = np.load(os.path.join(EMB_DIR, 'genre_embeddings.npy'))  # (n_genres, 8)
lang_weights  = np.load(os.path.join(EMB_DIR, 'lang_embeddings.npy'))   # (n_langs,  4)

with open(os.path.join(EMB_DIR, 'genre_encoder.json')) as f:
    genre2idx = json.load(f)
with open(os.path.join(EMB_DIR, 'lang_encoder.json')) as f:
    lang2idx  = json.load(f)

print(f"Genre embedding matrix loaded: {genre_weights.shape}")
print(f"Lang  embedding matrix loaded: {lang_weights.shape}")


In [ ]:
def lookup_lang_embeddings(series):
    """
    Map each language string to its embedding vector.
    Returns array of shape (n_rows, lang_emb_dim).
    """
    indices = series.fillna('<UNK>').map(lambda x: lang2idx.get(x, 0)).values
    return lang_weights[indices]   # fancy indexing — each row gets its vector


def lookup_genre_embeddings(series):
    """
    For each row, split genres by '|', look up each token's embedding,
    and mean-pool across tokens.
    Returns array of shape (n_rows, genre_emb_dim).
    """
    result = []
    for g_str in series.fillna('<UNK>'):
        tokens  = g_str.split('|')
        indices = [genre2idx.get(t, 0) for t in tokens]
        # stack the individual genre vectors and mean across them
        vecs    = genre_weights[indices]   # (n_tokens, emb_dim)
        result.append(vecs.mean(axis=0))   # (emb_dim,)
    return np.array(result)


def build_embedding_matrix(df_tr, df_v, df_te):
    """
    Build dense feature matrices using learned embeddings for categoricals.
    Returns X_train, X_val, X_test as plain numpy arrays.
    """
    def _build(df_):
        num_part   = df_[NUM_COLS].values.astype(np.float32)
        lang_part  = lookup_lang_embeddings(df_[CAT_LANG])
        genre_part = lookup_genre_embeddings(df_[CAT_GENRE])
        return np.concatenate([num_part, lang_part, genre_part], axis=1)

    return _build(df_tr), _build(df_v), _build(df_te)

X_train_emb, X_val_emb, X_test_emb = build_embedding_matrix(
    df_train, df_val, df_test
)
genre_emb_dim = genre_weights.shape[1]
lang_emb_dim  = lang_weights.shape[1]

print(f"Embedding feature matrix — train: {X_train_emb.shape}")
print(f"  {len(NUM_COLS)} numeric + {lang_emb_dim} lang dims + {genre_emb_dim} genre dims "
      f"= {X_train_emb.shape[1]} total features")


## 7. Train All 6 Classical Models

We train each of the 3 algorithms twice — once with baseline features, once with embedding
features — keeping hyperparameters identical between the two regimes so any difference
in performance is purely attributable to the feature encoding.


In [ ]:
def evaluate(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    return rmse, mae


def train_and_evaluate(model_obj, X_tr, X_v, X_te, y_tr, y_v, y_te, label):
    """Fit model, evaluate on all splits, print results, return result dict."""
    t0 = time.perf_counter()
    model_obj.fit(X_tr, y_tr)
    train_time = time.perf_counter() - t0

    tr_rmse, tr_mae = evaluate(y_tr,  model_obj.predict(X_tr))
    v_rmse,  v_mae  = evaluate(y_v,   model_obj.predict(X_v))
    te_rmse, te_mae = evaluate(y_te,  model_obj.predict(X_te))

    print(f"\n{'─'*55}")
    print(f"  {label}")
    print(f"{'─'*55}")
    print(f"  Train      RMSE: {tr_rmse:.4f}   MAE: {tr_mae:.4f}")
    print(f"  Validation RMSE: {v_rmse:.4f}   MAE: {v_mae:.4f}")
    print(f"  Test       RMSE: {te_rmse:.4f}   MAE: {te_mae:.4f}")
    print(f"  Train time : {train_time:.1f}s")

    return {
        'model': label, 'train_rmse': tr_rmse, 'val_rmse': v_rmse,
        'test_rmse': te_rmse, 'train_mae': tr_mae, 'val_mae': v_mae,
        'test_mae': te_mae, 'train_time_s': round(train_time, 1)
    }


In [ ]:
# ── Model definitions — same hyperparameters for both regimes ──
def make_models():
    return {
        'LinearRegression': LinearRegression(),
        'RandomForest': RandomForestRegressor(
            n_estimators=200, max_depth=12,
            min_samples_leaf=5, random_state=42, n_jobs=-1
        ),
        'XGBoost': XGBRegressor(
            n_estimators=300, max_depth=6, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8,
            tree_method='hist', random_state=42, verbosity=0
        )
    }

results = []

# ── Baseline regime ──
print("\n========== REGIME A: BASELINE (OHE / multi-hot) ==========")
for name, m in make_models().items():
    r = train_and_evaluate(
        m,
        X_train_base, X_val_base, X_test_base,
        y_train, y_val, y_test,
        label=f"{name} [Baseline]"
    )
    results.append(r)

# ── Embedding regime ──
print("\n========== REGIME B: LEARNED EMBEDDINGS ==========")
for name, m in make_models().items():
    r = train_and_evaluate(
        m,
        X_train_emb, X_val_emb, X_test_emb,
        y_train, y_val, y_test,
        label=f"{name} [Embeddings]"
    )
    results.append(r)


## 8. Load ANN Result from NB1

In [ ]:
with open(os.path.join(EMB_DIR, 'ann_results.json')) as f:
    ann_r = json.load(f)
results.append(ann_r)
print("ANN result loaded:", ann_r)


## 9. Final Comparison Table

In [ ]:
results_df = pd.DataFrame(results)[[
    'model', 'train_rmse', 'val_rmse', 'test_rmse',
    'train_mae', 'val_mae', 'test_mae', 'train_time_s'
]]
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_colwidth', 35)

print("=" * 90)
print("FINAL COMPARISON — ALL 7 MODELS")
print("=" * 90)
print(results_df.to_string(index=False))
print("=" * 90)
print("\nLower RMSE / MAE = better.  Compare [Baseline] vs [Embeddings] rows per algorithm.")


In [ ]:
# ── Bar chart: Test RMSE comparison ──
fig, ax = plt.subplots(figsize=(12, 5))
colors = []
for m in results_df['model']:
    if 'ANN' in m:       colors.append('#534AB7')
    elif 'Embed' in m:   colors.append('#1D9E75')
    else:                colors.append('#888780')

bars = ax.barh(results_df['model'], results_df['test_rmse'], color=colors)
ax.set_xlabel('Test RMSE  (lower is better)')
ax.set_title('Test RMSE — Baseline vs Learned Embeddings vs ANN')
ax.axvline(results_df['test_rmse'].min(), color='red', linestyle='--', linewidth=0.8,
           label=f"Best: {results_df['test_rmse'].min():.4f}")
ax.legend()

# Value labels on bars
for bar, val in zip(bars, results_df['test_rmse']):
    ax.text(val + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#534AB7', label='EmbeddingANN'),
    Patch(facecolor='#1D9E75', label='Classical + Embeddings'),
    Patch(facecolor='#888780', label='Classical Baseline'),
]
ax.legend(handles=legend_elements, loc='lower right')
plt.tight_layout()
plt.savefig(os.path.join(EMB_DIR, 'comparison_chart.png'), dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Delta table: how much did embeddings help each classical model? ──
baseline_df  = results_df[results_df['model'].str.contains('Baseline')].reset_index(drop=True)
embedding_df = results_df[results_df['model'].str.contains('Embeddings')].reset_index(drop=True)

delta_df = pd.DataFrame({
    'Algorithm'  : ['LinearRegression', 'RandomForest', 'XGBoost'],
    'RMSE Baseline'   : baseline_df['test_rmse'].values,
    'RMSE Embeddings' : embedding_df['test_rmse'].values,
})
delta_df['Delta RMSE'] = delta_df['RMSE Embeddings'] - delta_df['RMSE Baseline']
delta_df['Improvement'] = delta_df['Delta RMSE'].map(
    lambda x: f"{'better' if x < 0 else 'worse'} by {abs(x):.4f}"
)

print("\nImpact of learned embeddings on each classical model:")
print(delta_df.to_string(index=False))
